In [ ]:
import random
import time

import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from ugot import ugot

got = ugot.UGOT()
got.initialize("192.168.88.1")
got.show_light_rgb([1, 2, 3], 255, 255, 255)

In [ ]:
got.load_models(['gesture', 'apriltag_qrcode'])

In [ ]:
directions = {
    "Stone": "forward",
    "Cloth": "backward",
    "Point Left": "left",
    "Point Right": "right",
}


def classify_pose(angles, index_direction):
    # Finger order: index, middle, ring, little.
    extended = [angle > 140 for angle in angles]
    folded = [angle < 110 for angle in angles]

    # shortcut: heuristic thresholds; calibrate using your camera and hands.
    if all(folded):
        return "Stone"

    # Allow slight bends and landmark noise in an open hand.
    if all(extended):
        return "Cloth"

    if extended[0] and all(folded[1:]):
        dx, dy = index_direction
        # Use the visible finger direction on the unmirrored camera feed.
        if abs(dx) > abs(dy):
            return "Point Right" if dx < 0 else "Point Left"

    return None


def recognize_gesture(landmarks, image_points):
    points = np.array([
        [point.x, point.y, point.z] for point in landmarks
    ])

    # Measure how straight each finger is at its middle joint.
    bases = points[[5, 9, 13, 17]]
    joints = points[[6, 10, 14, 18]]
    tips = points[[8, 12, 16, 20]]

    a = bases - joints
    b = tips - joints
    lengths = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
    palm_size = np.linalg.norm(points[0] - points[9])

    if np.any(lengths < 1e-8) or palm_size < 1e-8:
        return None

    cosines = np.sum(a * b, axis=1) / lengths
    angles = np.degrees(np.arccos(np.clip(cosines, -1, 1)))

    index_direction = np.subtract(image_points[8], image_points[5])
    return classify_pose(angles, index_direction)

# Finger chains and the connections across the palm.
hand_chains = (
    (0, 1, 2, 3, 4), (0, 5, 6, 7, 8), (5, 9, 10, 11, 12),
    (9, 13, 14, 15, 16), (13, 17, 18, 19, 20), (0, 17),
)

options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(
        model_asset_path="hand_landmarker.task"
    ),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,  # Detect a second hand so ambiguous control can stop.
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.7,
    min_tracking_confidence=0.7,
)

last_timestamp = -1
cap = cv2.VideoCapture(0)

try:
    if not cap.isOpened():
        raise RuntimeError("Could not open webcam")
    
    with vision.HandLandmarker.create_from_options(options) as landmarker:
        while True:
            ret, frame = cap.read()
            
            if not ret:
                print("Camera frame unavailable. Stopping program")
                break
            
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb,
            )

            timestamp = max(
                last_timestamp + 1,
                time.monotonic_ns() // 1_000_000,
            )
            last_timestamp = timestamp

            result = landmarker.detect_for_video(mp_image, timestamp)

            gesture_result = None
            height, width = frame.shape[:2]
            for hand_index, landmarks in enumerate(result.hand_landmarks):
                image_points = [
                    (int(point.x * width), int(point.y * height))
                    for point in landmarks
                ]
                for chain in hand_chains:
                    for start, end in zip(chain, chain[1:]):
                        cv2.line(frame, image_points[start], image_points[end],
                                 (0, 255, 0), 2)
                for point in image_points:
                    cv2.circle(frame, point, 4, (0, 0, 255), -1)

                if len(result.hand_world_landmarks) == 1:
                    gesture_result = recognize_gesture(
                        result.hand_world_landmarks[hand_index], image_points
                    )

            direction = directions.get(gesture_result, "stop")
            label = f"{gesture_result or 'Unknown'}: {direction}"

            cv2.putText(
                frame, label, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2,
            )
            cv2.imshow("Camera Feed", frame)

            if cv2.waitKey(10) & 0xFF == ord("q"):
                break

            if direction == "forward":
                got.mecanum_move_speed(0, 20)
            elif direction == "backward":
                got.mecanum_move_speed(1, 20)
            elif direction == "left":
                got.mecanum_turn_speed(2, 20)
            elif direction == "right":
                got.mecanum_turn_speed(3, 20)
            else:
                got.mecanum_stop()

except KeyboardInterrupt:
    print("Remote control ended.")

finally:
    try:
        got.mecanum_stop()
    finally:
        cap.release()
        cv2.destroyAllWindows()

In [ ]:
def pickup():
  got.mechanical_joint_control(0, 0, 0, 500)
  time.sleep(1)
  got.mechanical_clamp_release()
  time.sleep(1)
  got.mechanical_joint_control(0, -40, -10, 500)
  time.sleep(1)
  got.mechanical_clamp_close()
  time.sleep(1)
  got.mechanical_joint_control(0, 45, 45, 500)

pickup()

In [ ]:
def release():
  got.mechanical_joint_control(0, -40, -10, 500)
  time.sleep(1)
  got.mechanical_clamp_release()
  time.sleep(1)
  got.mechanical_joint_control(0, 45, 45, 500)
  time.sleep(1)
  got.mechanical_clamp_close()
  time.sleep(1)

release()